# Workspace Migration

This notebook helps migrate content between Microsoft Fabric workspaces.

## What this notebook does:
- Lists all workspaces
- Shows workspace contents (datasets, reports, dashboards)
- Creates migration plans
- Helps move content between workspaces

In [ ]:
# Import required modules
import sys
sys.path.append('..')

from modules.fabric_auth import FabricAuth, load_credentials
from modules.fabric_client import FabricClient, workspaces_to_dataframe
from modules.utils import create_migration_plan, export_to_excel
import pandas as pd
import json

In [ ]:
# Load credentials and authenticate
credentials = load_credentials('../config/credentials.json')
auth = FabricAuth(
    tenant_id=credentials['tenant_id'],
    client_id=credentials['client_id'],
    client_secret=credentials['client_secret']
)

token = auth.get_access_token()
if token:
    print("✓ Authentication successful")
    client = FabricClient(token)
else:
    print("✗ Authentication failed")

In [ ]:
# Get all workspaces
print("Fetching workspaces...")
workspaces = client.get_workspaces()

if workspaces:
    print(f"✓ Found {len(workspaces)} workspaces")
    workspaces_df = workspaces_to_dataframe(workspaces)
    display(workspaces_df[['id', 'name', 'type', 'state']].head(20))
else:
    print("✗ No workspaces found or error occurred")

In [ ]:
# Select source workspace (modify the workspace_id)
source_workspace_id = "YOUR_SOURCE_WORKSPACE_ID"  # Replace with actual ID

# Get workspace details
print(f"\nFetching details for source workspace...")
source_workspace = client.get_workspace(source_workspace_id)

if source_workspace:
    print(f"✓ Source workspace: {source_workspace.get('name', 'Unknown')}")
else:
    print("✗ Could not fetch source workspace")

In [ ]:
# Get content from source workspace
if source_workspace:
    print("\n=== Source Workspace Content ===")
    
    # Get datasets
    datasets = client.get_datasets(source_workspace_id)
    if datasets:
        print(f"\nDatasets: {len(datasets)}")
        datasets_df = pd.DataFrame(datasets)
        display(datasets_df[['id', 'name']]) if 'name' in datasets_df.columns else display(datasets_df)
    
    # Get reports
    reports = client.get_reports(source_workspace_id)
    if reports:
        print(f"\nReports: {len(reports)}")
        reports_df = pd.DataFrame(reports)
        display(reports_df[['id', 'name']]) if 'name' in reports_df.columns else display(reports_df)
    
    # Get dashboards
    dashboards = client.get_dashboards(source_workspace_id)
    if dashboards:
        print(f"\nDashboards: {len(dashboards)}")
        dashboards_df = pd.DataFrame(dashboards)
        display(dashboards_df[['id', 'displayName']]) if 'displayName' in dashboards_df.columns else display(dashboards_df)

In [ ]:
# Select target workspace
target_workspace_id = "YOUR_TARGET_WORKSPACE_ID"  # Replace with actual ID

# Get workspace details
print(f"\nFetching details for target workspace...")
target_workspace = client.get_workspace(target_workspace_id)

if target_workspace:
    print(f"✓ Target workspace: {target_workspace.get('name', 'Unknown')}")
else:
    print("✗ Could not fetch target workspace")

In [ ]:
# Create migration plan
# Select items to migrate (modify as needed)
items_to_migrate = []

# Add dataset IDs to migrate
if datasets:
    # Example: migrate first dataset
    # items_to_migrate.append(datasets[0]['id'])
    pass

# Create migration plan
migration_plan = create_migration_plan(
    source_workspace_id=source_workspace_id,
    target_workspace_id=target_workspace_id,
    items_to_migrate=items_to_migrate
)

print("\n=== Migration Plan ===")
print(json.dumps(migration_plan, indent=2))

In [ ]:
# Export migration plan
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Save migration plan as JSON
plan_filename = f"migration_plan_{timestamp}.json"
with open(plan_filename, 'w') as f:
    json.dump(migration_plan, f, indent=2)

print(f"\n✓ Migration plan saved to {plan_filename}")

# Create Excel report with workspace contents
excel_filename = f"workspace_migration_analysis_{timestamp}.xlsx"
with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
    if datasets:
        datasets_df.to_excel(writer, sheet_name='Source Datasets', index=False)
    if reports:
        reports_df.to_excel(writer, sheet_name='Source Reports', index=False)
    if dashboards:
        dashboards_df.to_excel(writer, sheet_name='Source Dashboards', index=False)

print(f"✓ Migration analysis exported to {excel_filename}")

## Notes on Migration

⚠️ **Important**: 
- This notebook helps analyze and plan migrations
- Actual content migration may require additional API calls or Power BI REST API operations
- Some content types may need to be republished rather than migrated
- Always test migrations in a non-production environment first
- Consider dependencies between datasets, reports, and dashboards